# SQL Case Study

## Part 1: phpMyAdmin (Q1-Q9)

### Q1: Some of the facilities charge a fee to members, but some do not. Write a SQL query to produce a list of the names of the facilities that do.

In [ ]:
SELECT * 
FROM Facilities 
WHERE membercost > 0;

### Q2: How many facilities do not charge a fee to members?

In [ ]:
SELECT COUNT(*) 
FROM Facilities 
WHERE membercost = 0;

### Q3: Write an SQL query to show a list of facilities that charge a fee to members, where the fee is less than 20% of the facility's monthly maintenance cost. Return the facid, facility name, member cost, and monthly maintenance of the facilities in question.

In [ ]:
SELECT 
  facid,
  name,
  membercost,
  monthlymaintenance * 0.2 AS maintenance_20_percent
FROM Facilities
WHERE membercost > 0
  AND membercost < (monthlymaintenance * 0.2);

### Q4: Write an SQL query to retrieve the details of facilities with ID 1 and 5. Try writing the query without using the OR operator.

In [ ]:
SELECT * 
FROM Facilities 
WHERE facid IN (1, 5);

### Q5: Produce a list of facilities, with each labelled as'cheap' or 'expensive', depending on if their monthly maintenance cost is more than $100. Return the name and monthly maintenance of the facilities in question.

In [ ]:
SELECT 
	*,
    monthlymaintenance > 100 AS expensive,
    monthlymaintenance <= 100 AS cheap
FROM Facilities;

### Q6: You'd like to get the first and last name of the last member(s) who signed up. Try not to use the LIMIT clause for your solution.

In [ ]:
SELECT 
	firstname, 
    surname
FROM Members
WHERE joindate = (
    SELECT MAX(joindate)
    FROM Members
);

### Q7: Produce a list of all members who have used a tennis court. Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name.

In [ ]:
SELECT DISTINCT
	Members.firstname AS first_name,
    Members.surname AS last_name,
    Facilities.name AS facility_name
FROM Bookings 
INNER JOIN Members 
USING(memid)
INNER JOIN Facilities
USING(facid)
WHERE Facilities.name LIKE 'Tennis%'
ORDER BY first_name, last_name;

### Q8: Produce a list of bookings on the day of 2012-09-14 which will cost the member (or guest) more than $30. Remember that guests have different costs to members (the listed costs are per half-hour 'slot'), and the guest user's ID is always 0. Include in your output the name of the facility, the name of the member formatted as a single column, and the cost. Order by descending cost, and do not use any subqueries.

In [ ]:
SELECT 
    CASE
        WHEN memid = 0 THEN 'Guest'
        ELSE CONCAT_WS(' ', Members.firstname, Members.surname)
    END AS member_name,
    Facilities.name AS facility_name,
    CASE 
    	WHEN memid = 0 THEN slots * guestcost 
        ELSE slots * membercost 
	END AS cost
FROM Bookings 
INNER JOIN Facilities USING(facid) 
INNER JOIN Members USING(memid) 
WHERE starttime >= '2012-09-14 00:00:00' 
	AND starttime < '2012-09-15 00:00:00'
HAVING cost > 30
ORDER BY cost DESC;

### Q9: This time, produce the same result as in Q8, but using a subquery.

In [ ]:
SELECT
    member_name,
    facility_name,
    cost
FROM (
    SELECT 
        CASE
            WHEN memid = 0 THEN 'Guest'
            ELSE CONCAT_WS(' ', Members.firstname, Members.surname)
        END AS member_name,
        Facilities.name AS facility_name,
        CASE 
            WHEN memid = 0 THEN slots * guestcost 
            ELSE slots * membercost 
        END AS cost
    FROM Bookings 
    INNER JOIN Facilities USING (facid) 
    INNER JOIN Members USING (memid) 
    WHERE starttime >= '2012-09-14 00:00:00' 
      AND starttime <  '2012-09-15 00:00:00'
) AS booking_costs
WHERE cost > 30
ORDER BY cost DESC;

## Part 2: Jupyter / SQLite (Q10-Q13)

### Q10: Produce a list of facilities with a total revenue less than 1000. The output of facility name and total revenue, sorted by revenue. Remember that there's a different cost for guests and members!

In [ ]:
SELECT
    Facilities.name AS facility_name,
    SUM(
        CASE
            WHEN Bookings.memid = 0 THEN Bookings.slots * Facilities.guestcost
            ELSE Bookings.slots * Facilities.membercost
        END
    ) AS total_revenue
FROM Facilities
JOIN Bookings
  ON Bookings.facid = Facilities.facid
GROUP BY Facilities.facid, Facilities.name
HAVING total_revenue < 1000
ORDER BY total_revenue;

### Q11: Produce a report of members and who recommended them in alphabetic surname,firstname order

In [ ]:
SELECT
    member.surname,
    member.firstname,
    recommended_by.surname AS recommender_surname,
    recommended_by.firstname AS recommender_firstname
FROM Members AS member
LEFT JOIN Members AS recommended_by
  ON member.recommendedby = recommended_by.memid
ORDER BY member.surname, member.firstname;

### Q12: Find the facilities with their usage by member, but not guests

In [ ]:
SELECT
    facility.name AS facility_name,
    SUM(bookings.slots) AS member_usage
FROM Facilities AS facility
JOIN Bookings AS bookings
  ON bookings.facid = facility.facid
WHERE bookings.memid != 0
GROUP BY facility.facid, facility.name
ORDER BY member_usage DESC;

### Q13: Find the facilities usage by month, but not guests

In [ ]:
SELECT
    facility.name AS facility_name,
    DATE_FORMAT(bookings.starttime, '%Y-%m') AS month,
    SUM(bookings.slots) AS member_usage
FROM Facilities AS facility
JOIN Bookings AS bookings
  ON bookings.facid = facility.facid
WHERE bookings.memid != 0
GROUP BY facility.facid, facility.name, DATE_FORMAT(bookings.starttime, '%Y-%m')
ORDER BY facility.name, month;